# Stage 5: Rolling/Expanding Window Backtest

Train only on past data and evaluate future periods using cached modeling table.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from tqdm import tqdm

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data' / 'intermediate'

df = pd.read_parquet(DATA_DIR / 'model_table.parquet').sort_index()
df = df.dropna(subset=['ret_t1'])

TRAD_FEATS = ['momentum', 'value_composite', 'quality_composite']
SENT_FEATS = ['finbert_sent_7d', 'finbert_sent_30d', 'sent_momentum',
              'gdelt_tone_7d', 'gdelt_tone_30d', 'news_volume_7d', 'news_volume_30d']
ALL_FEATS = TRAD_FEATS + SENT_FEATS

print('Rows:', len(df), 'Date range:', df.index.get_level_values('date').min(), '->', df.index.get_level_values('date').max())

In [ ]:
def _clip_and_prepare(X, q1, q9, fill_na_feats, imputer=None):
    """
    Clip to [q1, q9], then either:
      - fill_na_feats=True : mean-impute NaN using a SimpleImputer fitted on training data.
        Pass imputer=None to fit a new one; pass an existing imputer to transform only.
      - fill_na_feats=False: drop rows with any NaN.
    Returns (X_arr as np.ndarray, imputer_or_None, row_index).
    """
    X = X.clip(lower=q1, upper=q9, axis=1)
    if fill_na_feats:
        if imputer is None:
            imputer = SimpleImputer(strategy='mean')
            X_arr = imputer.fit_transform(X.values)
        else:
            X_arr = imputer.transform(X.values)
        X_arr = np.nan_to_num(X_arr)   # safety: columns all-NaN → imputer gives NaN → map to 0
        return X_arr, imputer, X.index
    else:
        X = X.dropna()
        return X.values, None, X.index


def select_alpha_expanding(
    train_panel,
    feature_cols,
    alpha_grid=None,
    fill_na_feats=False,
    min_train_rows=10000,
    val_days=252,
    min_val_rows=1000,
    min_inner_train_rows=5000,
    metric='spearman',
):
    if alpha_grid is None:
        alpha_grid = [0.1, 0.5, 1.0, 5.0, 10.0, 25.0, 50.0]

    train_dates = train_panel.index.get_level_values('date').unique().sort_values()
    if len(train_dates) < val_days + 1:
        return alpha_grid[0], pd.DataFrame()

    val_start = train_dates[-val_days]
    inner_train = train_panel.loc[train_panel.index.get_level_values('date') < val_start].copy()
    val_panel   = train_panel.loc[train_panel.index.get_level_values('date') >= val_start].copy()

    inner_train = inner_train.dropna(subset=['ret_t1'] + TRAD_FEATS)
    if len(inner_train) < min_inner_train_rows or len(val_panel) < min_val_rows:
        return alpha_grid[0], pd.DataFrame()

    X_train = inner_train[feature_cols].copy()
    q1 = X_train.quantile(0.01)
    q9 = X_train.quantile(0.99)

    # FIX: use mean imputation instead of fillna(0) so the scaler sees the true
    # feature distribution, not one distorted by ~99% artificial zeros.
    X_train_arr, imputer, train_idx = _clip_and_prepare(X_train, q1, q9, fill_na_feats)
    if len(X_train_arr) == 0:
        return alpha_grid[0], pd.DataFrame()

    y_train = inner_train.loc[train_idx, 'ret_t1'].values

    X_val = val_panel[feature_cols].copy()
    # Re-use the imputer fitted on inner_train so there is no data leakage
    X_val_arr, _, val_idx = _clip_and_prepare(X_val, q1, q9, fill_na_feats, imputer=imputer)
    if len(X_val_arr) == 0:
        return alpha_grid[0], pd.DataFrame()

    y_val = val_panel.loc[val_idx, 'ret_t1'].copy()
    if y_val.empty:
        return alpha_grid[0], pd.DataFrame()

    scaler = StandardScaler()
    Xs_train = scaler.fit_transform(X_train_arr)
    Xs_val   = scaler.transform(X_val_arr)

    scores = []
    for alpha in alpha_grid:
        model = Ridge(alpha=alpha)
        model.fit(Xs_train, y_train)
        pred = pd.Series(model.predict(Xs_val), index=val_idx)
        joined = pd.DataFrame({'pred': pred, 'ret_t1': y_val}).dropna()
        if joined.empty:
            score = np.nan
        elif metric == 'mse':
            score = -((joined['pred'] - joined['ret_t1']) ** 2).mean()
        else:
            daily_ic = joined.groupby(level='date').apply(
                lambda g: g['pred'].corr(g['ret_t1'], method='spearman')
            )
            score = daily_ic.dropna().mean()
        scores.append({'alpha': alpha, 'score': score})

    score_df = pd.DataFrame(scores).sort_values(['score', 'alpha'], ascending=[False, True])
    if score_df['score'].notna().any():
        best_alpha = float(score_df.iloc[0]['alpha'])
    else:
        best_alpha = alpha_grid[0]

    return best_alpha, score_df


def prescan_best_alpha(
    panel,
    feature_cols,
    alpha_grid=None,
    fill_na_feats=False,
    n_scans=5,
    val_days=252,
    metric='spearman',
    label='prescan',
):
    """
    Run select_alpha_expanding at n_scans evenly-spaced expanding-window cutoffs
    covering the full dataset, average the per-alpha IC across splits, and return
    the alpha with the best mean score.

    This is the correct way to pick alpha for an expanding-window strategy:
    evaluate every candidate across multiple out-of-sample windows, then adopt
    the single best alpha for the final backtest.
    """
    if alpha_grid is None:
        alpha_grid = [0.1, 0.5, 1.0, 5.0, 10.0, 25.0, 50.0]

    dates = panel.index.get_level_values('date').unique().sort_values()
    start_idx = len(dates) // 3          # need enough training data before the first split
    end_idx   = len(dates) - val_days - 5
    if end_idx <= start_idx:
        return alpha_grid[len(alpha_grid) // 2]

    cutoff_indices = np.linspace(start_idx, end_idx, n_scans, dtype=int)

    all_score_series = []
    for ci in cutoff_indices:
        # sub_panel covers data up to (but not past) dates[ci + val_days]
        window_end = dates[min(ci + val_days, len(dates) - 1)]
        sub_panel  = panel.loc[panel.index.get_level_values('date') <= window_end]
        _, score_df = select_alpha_expanding(
            sub_panel, feature_cols,
            alpha_grid=alpha_grid, fill_na_feats=fill_na_feats,
            val_days=val_days,
        )
        if not score_df.empty and score_df['score'].notna().any():
            all_score_series.append(score_df.set_index('alpha')['score'])

    if not all_score_series:
        fallback = alpha_grid[len(alpha_grid) // 2]
        print(f'  [{label}] no valid splits found; falling back to alpha={fallback}')
        return fallback

    avg_scores = pd.concat(all_score_series, axis=1).mean(axis=1)
    best_alpha = float(avg_scores.idxmax())
    print(f'  [{label}] avg IC per alpha:\n{avg_scores.round(6).to_string()}')
    print(f'  [{label}] => best alpha = {best_alpha}')
    return best_alpha


def run_walkforward(
    panel,
    feature_cols,
    rebal_freq=5,
    train_years=3,
    expanding=False,
    n_long=50,
    n_short=50,
    alpha=5.0,
    tune_alpha=False,
    alpha_grid=None,
    cost_bps=10,
    fill_na_feats=False,
    min_train_rows=10000,
    label='walkforward',
    prescan_alpha=False,
    prescan_n_scans=5,
):
    """
    Walk-forward expanding/rolling-window backtest with Ridge regression.

    Alpha selection modes (mutually exclusive; prescan_alpha takes priority):
      prescan_alpha=True  : run prescan_best_alpha once before the main loop to
                            find the globally best alpha across n_scans expanding
                            validation windows, then use that fixed alpha for every
                            rebalancing date.  (Recommended)
      tune_alpha=True     : re-select alpha at every rebalancing date via a single
                            inner expanding-window split. (Legacy, noisier)
      neither             : use the fixed `alpha` argument throughout.
    """
    dates       = panel.index.get_level_values('date').unique().sort_values()
    rebal_dates = dates[np.arange(0, len(dates), rebal_freq)]

    port         = pd.Series(index=dates, dtype=float)
    last_w       = pd.Series(dtype=float)
    alpha_history = []

    # --- Pre-scan alpha once for the full dataset ---
    if prescan_alpha:
        if alpha_grid is None:
            alpha_grid = [0.1, 0.5, 1.0, 5.0, 10.0, 25.0, 50.0]
        alpha = prescan_best_alpha(
            panel, feature_cols,
            alpha_grid=alpha_grid,
            fill_na_feats=fill_na_feats,
            n_scans=prescan_n_scans,
            label=label,
        )
        print(f'  [{label}] using fixed alpha={alpha} for all rebalancing dates')

    for d in tqdm(rebal_dates, desc=label, leave=False):
        idx_dates = panel.index.get_level_values('date')
        if expanding:
            train_mask = idx_dates < d
        else:
            train_start = d - pd.DateOffset(years=train_years)
            train_mask  = (idx_dates >= train_start) & (idx_dates < d)

        train = panel.loc[train_mask].dropna(subset=['ret_t1'] + TRAD_FEATS)
        if len(train) < min_train_rows:
            continue

        chosen_alpha = alpha
        if not prescan_alpha and tune_alpha:
            chosen_alpha, alpha_scores = select_alpha_expanding(
                train, feature_cols,
                alpha_grid=alpha_grid,
                fill_na_feats=fill_na_feats,
                min_train_rows=min_train_rows,
            )
            alpha_history.append({
                'date': d,
                'alpha': chosen_alpha,
                'n_train_rows': len(train),
                'n_alpha_candidates': len(alpha_grid) if alpha_grid is not None else 7,
                'best_score': alpha_scores['score'].max() if not alpha_scores.empty else np.nan,
            })

        X = train[feature_cols].copy()
        q1 = X.quantile(0.01)
        q9 = X.quantile(0.99)

        # FIX: mean-impute so the scaler's std is not compressed by ~99% artificial zeros
        X_arr, imputer, train_idx = _clip_and_prepare(X, q1, q9, fill_na_feats)
        y = train.loc[train_idx, 'ret_t1'].values

        scaler = StandardScaler()
        Xs     = scaler.fit_transform(X_arr)
        model  = Ridge(alpha=chosen_alpha)
        model.fit(Xs, y)

        try:
            today = panel.loc[(d, slice(None)), feature_cols].copy()
        except KeyError:
            continue

        today_arr, _, today_idx = _clip_and_prepare(today, q1, q9, fill_na_feats, imputer=imputer)
        if len(today_arr) == 0:
            continue

        pred = model.predict(scaler.transform(today_arr))
        pred = pd.Series(pred, index=today_idx.get_level_values('ticker'))
        pred = pred.replace([np.inf, -np.inf], np.nan).dropna()

        longs  = pred.nlargest(n_long).index
        shorts = pred.nsmallest(n_short).index
        w      = pd.Series(0.0, index=pred.index)
        w.loc[longs]  =  1.0 / max(len(longs),  1)
        w.loc[shorts] = -1.0 / max(len(shorts), 1)
        gross = w.abs().sum()
        if gross > 0:
            w /= gross

        turnover = (w - last_w).abs().sum()
        cost     = turnover * (cost_bps / 10000.0)
        last_w   = w

        i        = np.where(rebal_dates == d)[0][0] + 1
        next_cut = rebal_dates[i] if i < len(rebal_dates) else dates[-1]
        span     = dates[(dates >= d) & (dates < next_cut)]

        for dd in span:
            r = panel.loc[(dd, w.index), 'ret_t1'].fillna(0.0)
            port.loc[dd] = (w * r).sum()
        if len(span) > 0:
            port.loc[span[0]] -= cost

    perf    = port.dropna()
    cum     = (1.0 + perf).cumprod()
    ann_ret = perf.mean() * 252
    ann_vol = perf.std(ddof=1) * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan
    max_dd  = (cum / cum.cummax() - 1.0).min()
    alpha_history = pd.DataFrame(alpha_history)
    return perf, cum, {'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe, 'max_dd': max_dd}, alpha_history

In [ ]:
from xgboost import XGBRegressor

# Shallow trees (max_depth=4) to avoid memorising individual stocks.
# NaN is handled natively — XGBoost learns the optimal default branch
# direction for missing values, so no imputation is needed.
XGB_PARAMS = dict(
    n_estimators      = 300,
    max_depth         = 4,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 100,
    reg_alpha         = 0.0,
    reg_lambda        = 1.0,
    tree_method       = 'hist',
    random_state      = 42,
    n_jobs            = -1,
)

def run_walkforward_xgb(
    panel,
    feature_cols,
    rebal_freq=5,
    train_years=3,
    expanding=False,
    n_long=50,
    n_short=50,
    cost_bps=10,
    min_train_rows=10000,
    xgb_params=None,
    label='xgb_walkforward',
):
    """
    Walk-forward backtest using XGBoost instead of Ridge.

    XGBoost handles NaN natively — it learns which branch missing-value rows
    should follow at each split. This means ALL_FEATS (including sparse
    sentiment) can be passed directly without imputation.
    """
    if xgb_params is None:
        xgb_params = XGB_PARAMS

    dates       = panel.index.get_level_values('date').unique().sort_values()
    rebal_dates = dates[np.arange(0, len(dates), rebal_freq)]

    port   = pd.Series(index=dates, dtype=float)
    last_w = pd.Series(dtype=float)

    for d in tqdm(rebal_dates, desc=label, leave=False):
        idx_dates = panel.index.get_level_values('date')
        if expanding:
            train_mask = idx_dates < d
        else:
            train_start = d - pd.DateOffset(years=train_years)
            train_mask  = (idx_dates >= train_start) & (idx_dates < d)

        train = panel.loc[train_mask].dropna(subset=['ret_t1'] + TRAD_FEATS)
        if len(train) < min_train_rows:
            continue

        X = train[feature_cols].copy()
        # Winsorise only — do NOT impute. XGBoost sees real NaN.
        q1 = X.quantile(0.01)
        q9 = X.quantile(0.99)
        X  = X.clip(lower=q1, upper=q9, axis=1)
        y  = train.loc[X.index, 'ret_t1'].values

        model = XGBRegressor(**xgb_params)
        model.fit(X.values, y)

        try:
            today = panel.loc[(d, slice(None)), feature_cols].copy()
        except KeyError:
            continue

        today = today.clip(lower=q1, upper=q9, axis=1)
        today = today.dropna(subset=TRAD_FEATS)
        if today.empty:
            continue

        pred = model.predict(today.values)
        pred = pd.Series(pred, index=today.index.get_level_values('ticker'))
        pred = pred.replace([np.inf, -np.inf], np.nan).dropna()

        longs  = pred.nlargest(n_long).index
        shorts = pred.nsmallest(n_short).index
        w      = pd.Series(0.0, index=pred.index)
        w.loc[longs]  =  1.0 / max(len(longs),  1)
        w.loc[shorts] = -1.0 / max(len(shorts), 1)
        gross = w.abs().sum()
        if gross > 0:
            w /= gross

        # blend 50/50 with previous weights to dampen XGBoost turnover
        if not last_w.empty:
            w_aligned, last_w_aligned = w.align(last_w, fill_value=0.0)
            w = 0.5 * w_aligned + 0.5 * last_w_aligned
            gross = w.abs().sum()
            if gross > 0:
                w /= gross

        turnover = (w - last_w.reindex(w.index, fill_value=0.0)).abs().sum()
        cost     = turnover * (cost_bps / 10000.0)
        last_w   = w

        i        = np.where(rebal_dates == d)[0][0] + 1
        next_cut = rebal_dates[i] if i < len(rebal_dates) else dates[-1]
        span     = dates[(dates >= d) & (dates < next_cut)]

        for dd in span:
            r = panel.loc[(dd, w.index), 'ret_t1'].fillna(0.0)
            port.loc[dd] = (w * r).sum()
        if len(span) > 0:
            port.loc[span[0]] -= cost

    perf    = port.dropna()
    cum     = (1.0 + perf).cumprod()
    ann_ret = perf.mean() * 252
    ann_vol = perf.std(ddof=1) * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan
    max_dd  = (cum / cum.cummax() - 1.0).min()
    return perf, cum, {'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe, 'max_dd': max_dd}

In [8]:
# Alpha grid for inner expanding-window validation
ALPHA_GRID = [0.1, 0.5, 1.0, 5.0, 10.0, 25.0, 50.0]
print('Alpha grid:', ALPHA_GRID)

Alpha grid: [0.1, 0.5, 1.0, 5.0, 10.0, 25.0, 50.0]


In [ ]:
ALPHA_GRID = [0.1, 0.5, 1.0, 5.0, 10.0, 25.0, 50.0]

perf_base_e, cum_base_e, stats_base_e, alpha_hist_base_e = run_walkforward(
    df,
    TRAD_FEATS,
    expanding=True,
    prescan_alpha=True,
    alpha_grid=ALPHA_GRID,
    fill_na_feats=False,
    label='baseline_expanding',
)

perf_aug_e, cum_aug_e, stats_aug_e, alpha_hist_aug_e = run_walkforward(
    df,
    ALL_FEATS,
    expanding=True,
    prescan_alpha=True,
    alpha_grid=ALPHA_GRID,
    fill_na_feats=True,
    label='augmented_expanding',
)

print('\nExpanding window with pre-scanned alpha')
print('Baseline:', stats_base_e)
print('Augmented:', stats_aug_e)

In [ ]:
summary = pd.DataFrame([
    {'scheme': 'expanding_tuned', 'model': 'baseline', **stats_base_e},
    {'scheme': 'expanding_tuned', 'model': 'augmented', **stats_aug_e},
])

alpha_summary = pd.concat([
    alpha_hist_base_e.assign(model='baseline'),
    alpha_hist_aug_e.assign(model='augmented'),
], ignore_index=True)

summary.to_parquet(DATA_DIR / 'backtest_summary.parquet', index=False)
summary.to_csv(DATA_DIR / 'backtest_summary.csv', index=False)
alpha_summary.to_parquet(DATA_DIR / 'alpha_tuning_summary.parquet', index=False)
alpha_summary.to_csv(DATA_DIR / 'alpha_tuning_summary.csv', index=False)

print(summary)
# alpha_summary is empty when prescan_alpha=True (no per-date tuning history)
if not alpha_summary.empty and 'alpha' in alpha_summary.columns:
    print(alpha_summary.groupby(['model', 'alpha']).size().rename('times_selected').reset_index())
else:
    print('  (prescan_alpha mode: single upfront alpha selection, no per-date history)')
print('Saved backtest_summary.parquet, backtest_summary.csv, alpha_tuning_summary.parquet, and alpha_tuning_summary.csv')

In [ ]:
# XGBoost expanding-window backtests
# Baseline: traditional features only
# Augmented: all features incl. sentiment — XGBoost routes NaN natively,
#            so no imputation is needed.

perf_xgb_base, cum_xgb_base, stats_xgb_base = run_walkforward_xgb(
    df,
    TRAD_FEATS,
    expanding=True,
    label='xgb_baseline',
)

perf_xgb_aug, cum_xgb_aug, stats_xgb_aug = run_walkforward_xgb(
    df,
    ALL_FEATS,
    expanding=True,
    label='xgb_augmented',
)

comparison = pd.DataFrame([
    {'model': 'Ridge  baseline',   **stats_base_e},
    {'model': 'Ridge  augmented',  **stats_aug_e},
    {'model': 'XGBoost baseline',  **stats_xgb_base},
    {'model': 'XGBoost augmented', **stats_xgb_aug},
])
comparison['ann_ret'] = (comparison['ann_ret'] * 100).round(2).astype(str) + '%'
comparison['ann_vol'] = (comparison['ann_vol'] * 100).round(2).astype(str) + '%'
comparison['sharpe']  = comparison['sharpe'].round(4)
comparison['max_dd']  = (comparison['max_dd']  * 100).round(2).astype(str) + '%'
print(comparison.to_string(index=False))

full_summary = pd.DataFrame([
    {'model': 'ridge_baseline',   **stats_base_e},
    {'model': 'ridge_augmented',  **stats_aug_e},
    {'model': 'xgb_baseline',     **stats_xgb_base},
    {'model': 'xgb_augmented',    **stats_xgb_aug},
])
full_summary.to_parquet(DATA_DIR / 'backtest_summary.parquet', index=False)
full_summary.to_csv(DATA_DIR / 'backtest_summary.csv', index=False)
print('\nSaved backtest_summary.parquet / .csv')